# 4. Análisis Predictivo - Machine Learning

**Objetivo:** entrenar y comparar distintos modelos de clasificación para predecir `Class`
(si la primera etapa del Falcon 9 aterriza con éxito) a partir de las features del lanzamiento
(masa de la carga, órbita, sitio de lanzamiento, booster, etc.).

**Flujo de trabajo:**

1. Cargar `dataset_part_2.csv` (salida del notebook `03_data_wrangling`).
2. Seleccionar las features y codificarlas con **one-hot encoding** (variables categóricas
   como `Orbit`, `LaunchSite`, `LandingPad`, `Serial`).
3. Estandarizar las features numéricas (`StandardScaler`).
4. Separar en train/test (80/20).
5. Entrenar 4 modelos con `GridSearchCV` (búsqueda de hiperparámetros con validación cruzada):
   **Regresión Logística, SVM, Árbol de Decisión y KNN**.
6. Evaluar cada modelo en el set de test: accuracy y matriz de confusión.
7. Comparar los 4 modelos y elegir el mejor.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

np.random.seed(2)

## Paso 1: cargar los datos

In [ ]:
df = pd.read_csv('../data/processed/dataset_part_2.csv')
df.head()

## Paso 2: features y one-hot encoding

Usamos columnas numéricas (`FlightNumber`, `PayloadMass`, `Flights`, `Block`, `ReusedCount`)
y booleanas (`GridFins`, `Reused`, `Legs`), y codificamos las categóricas (`Orbit`,
`LaunchSite`, `LandingPad`, `Serial`) con `pd.get_dummies`.

In [ ]:
features = ['FlightNumber', 'PayloadMass', 'Orbit', 'LaunchSite', 'Flights',
            'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial']

X = pd.get_dummies(df[features], columns=['Orbit', 'LaunchSite', 'LandingPad', 'Serial'])
X = X.astype('float64')

Y = df['Class'].to_numpy()

print(X.shape)
X.head()

> **Nota:** `Serial` identifica a cada booster individual. Codificarlo agrega una columna casi
> por lanzamiento, lo que puede favorecer el sobreajuste (el modelo podría "memorizar" boosters
> en vez de generalizar). Lo dejamos porque es la variable que usa la consigna del curso, pero es
> un buen punto para mencionar en la conclusión (posible mejora: sacar `Serial` y comparar).

## Paso 3: valores faltantes en las features

Algunas columnas numéricas pueden tener nulos (por ejemplo `Block`, que no está definido para
los boosters más viejos). Los modelos de scikit-learn no aceptan `NaN`, así que los completamos
con la media de cada columna.

In [ ]:
missing = X.isnull().sum()
missing[missing > 0]

In [ ]:
X = X.fillna(X.mean())
assert X.isnull().sum().sum() == 0

## Paso 4: estandarizar las features

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Paso 5: separar en train/test

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y, test_size=0.2, random_state=2)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Éxitos en test:", Y_test.sum(), "/", len(Y_test))

## Función auxiliar: matriz de confusión

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Fracaso (0)', 'Éxito (1)'],
                yticklabels=['Fracaso (0)', 'Éxito (1)'], ax=ax)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

## Modelo 1: Regresión Logística

In [ ]:
parameters_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs'],
}

lr = LogisticRegression(max_iter=1000)
logreg_cv = GridSearchCV(lr, parameters_lr, cv=10)
logreg_cv.fit(X_train, Y_train)

print("Mejores hiperparámetros:", logreg_cv.best_params_)
print("Mejor accuracy (CV):", round(logreg_cv.best_score_, 4))

In [ ]:
acc_lr = accuracy_score(Y_test, logreg_cv.predict(X_test))
print("Accuracy en test:", round(acc_lr, 4))
plot_confusion_matrix(Y_test, logreg_cv.predict(X_test), "Regresión Logística")

## Modelo 2: SVM

In [ ]:
parameters_svm = {
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'C': [0.01, 0.1, 1, 10],
    'gamma': ['scale', 'auto'],
}

svm = SVC()
svm_cv = GridSearchCV(svm, parameters_svm, cv=10)
svm_cv.fit(X_train, Y_train)

print("Mejores hiperparámetros:", svm_cv.best_params_)
print("Mejor accuracy (CV):", round(svm_cv.best_score_, 4))

In [ ]:
acc_svm = accuracy_score(Y_test, svm_cv.predict(X_test))
print("Accuracy en test:", round(acc_svm, 4))
plot_confusion_matrix(Y_test, svm_cv.predict(X_test), "SVM")

## Modelo 3: Árbol de Decisión

In [ ]:
parameters_tree = {
    'criterion': ['gini', 'entropy'],
    'splitter': ['best', 'random'],
    'max_depth': [2, 4, 6, 8, 10, None],
    'min_samples_leaf': [1, 2, 4],
    'min_samples_split': [2, 5, 10],
}

tree = DecisionTreeClassifier(random_state=2)
tree_cv = GridSearchCV(tree, parameters_tree, cv=10)
tree_cv.fit(X_train, Y_train)

print("Mejores hiperparámetros:", tree_cv.best_params_)
print("Mejor accuracy (CV):", round(tree_cv.best_score_, 4))

In [ ]:
acc_tree = accuracy_score(Y_test, tree_cv.predict(X_test))
print("Accuracy en test:", round(acc_tree, 4))
plot_confusion_matrix(Y_test, tree_cv.predict(X_test), "Árbol de Decisión")

## Modelo 4: KNN

In [ ]:
parameters_knn = {
    'n_neighbors': list(range(1, 11)),
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
    'p': [1, 2],
}

knn = KNeighborsClassifier()
knn_cv = GridSearchCV(knn, parameters_knn, cv=10)
knn_cv.fit(X_train, Y_train)

print("Mejores hiperparámetros:", knn_cv.best_params_)
print("Mejor accuracy (CV):", round(knn_cv.best_score_, 4))

In [ ]:
acc_knn = accuracy_score(Y_test, knn_cv.predict(X_test))
print("Accuracy en test:", round(acc_knn, 4))
plot_confusion_matrix(Y_test, knn_cv.predict(X_test), "KNN")

## Comparación final de modelos

In [ ]:
results = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'SVM', 'Árbol de Decisión', 'KNN'],
    'Accuracy CV (train)': [logreg_cv.best_score_, svm_cv.best_score_,
                             tree_cv.best_score_, knn_cv.best_score_],
    'Accuracy (test)': [acc_lr, acc_svm, acc_tree, acc_knn],
}).sort_values('Accuracy (test)', ascending=False).reset_index(drop=True)

results

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(results['Modelo'], results['Accuracy (test)'], color='#2980b9')
ax.set_ylabel('Accuracy en test')
ax.set_ylim(0, 1)
ax.set_title('Comparación de modelos - Accuracy en test')
for i, v in enumerate(results['Accuracy (test)']):
    ax.text(i, v + 0.02, f"{v:.2f}", ha='center')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

best_model = results.iloc[0]['Modelo']
print(f"Mejor modelo: {best_model}")

## Conclusión

*(completar con el análisis propio una vez corrido el notebook con los datos reales)*

- ¿Cuál fue el mejor modelo y por qué creés que le fue mejor con este problema?
- ¿Qué tan parecidos fueron los accuracy de CV (train) y de test? ¿Hay señales de overfitting?
- Mirando la matriz de confusión del mejor modelo: ¿comete más falsos positivos o falsos
  negativos? ¿Qué implicancia tiene eso para SpaceX (por ejemplo, al estimar el costo de un
  lanzamiento)?
- Ideas para mejorar: sacar `Serial` de las features, probar más hiperparámetros, sumar
  features nuevas (por ejemplo, features de la EDA con SQL / Folium), o probar otros modelos
  (Random Forest, Gradient Boosting).